In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_part(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/part.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
part = load_part(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

# Define promotion type filter
p_type_like = "PROMO"

# Build masks for ship date and promo parts
ship_mask = lineitem.L_SHIPDATE.between("1994-03-01", "1994-04-01", inclusive="left")
promo_keys = part.P_PARTKEY[part.P_TYPE.str.startswith(p_type_like)]

# Compute line-level revenue once
rev = lineitem.L_EXTENDEDPRICE * (1.0 - lineitem.L_DISCOUNT)

# Total revenue for the period
total_rev = rev[ship_mask].sum()

# Promo revenue for the same period
promo_rev = rev[ship_mask & lineitem.L_PARTKEY.isin(promo_keys)].sum()

# Final metric
total = promo_rev * 100 / total_rev